# Chapter 2: Defining Nonfunctional Requirements
*From: Designing Data-Intensive Applications (2nd Edition)*

Functional requirements describe **what** an app does — screens, buttons, the operations needed to fulfill its purpose. Nonfunctional requirements describe **how well** it does those things: fast, reliable, scalable, maintainable, secure, compliant. An app that is unbearably slow or unreliable might as well not exist, so these "invisible" requirements are every bit as load-bearing as the feature list.

This chapter walks the **four pillars** of nonfunctional requirements for data-intensive systems:

| Pillar              | One-line definition                                                   | Primary metric(s)                             |
|---------------------|-----------------------------------------------------------------------|-----------------------------------------------|
| **Performance**     | Fast response and sufficient throughput under expected load           | response-time percentiles (p50/p95/p99/p999), QPS |
| **Reliability**     | Continuing to work correctly even when things go wrong                | MTBF, availability ("nines"), SLO success rate |
| **Scalability**     | Ability to cope with increased load by adding resources               | linear scalability, cost per request at scale |
| **Maintainability** | Keeping the system easy to run, understand, and evolve over years     | change lead time, incident frequency, operator toil |

> "Good operations can often work around the limitations of bad software, but good software cannot run reliably with bad operations." — Jay Kreps

## TL;DR

- **Timelines case study** frames the chapter: 500M posts/day (≈5,800/s), 10M concurrent users, avg 200 followers — a pull-model is 400M lookups/s; a push (fan-out-on-write) model is ~1.16M writes/s. Hybrid handles celebrities specially.
- **Response time is a distribution, not a number.** Use p50/p95/p99/p999, never the mean. Mean hides the tail. Averaging percentiles is mathematically meaningless.
- **Tail-latency amplification**: if any backend call has p99 = 1s, a request fanning out to 100 backends is slow ~63% of the time — even though each individual call is fast 99% of the time.
- **Metastable failure**: overloaded systems can enter a retry-storm feedback loop that persists after load subsides. Defenses: exponential backoff, circuit breakers, token buckets, load shedding, backpressure.
- **Reliability** = fault tolerance. Distinguish a *fault* (a part malfunctions) from a *failure* (whole system stops). Eliminate SPOFs; use redundancy, chaos engineering, blameless postmortems.
- **Scaling** spans shared-memory (vertical), shared-disk (NAS/SAN), and shared-nothing (horizontal). Cloud-native systems increasingly separate storage from compute.
- **Maintainability** = operability + simplicity + evolvability. Most software cost is maintenance, not initial dev. Minimize irreversibility.

## 1. Case Study: Social Network Home Timelines

### Requirements

**Functional**
- Users can post short messages
- Users can follow and unfollow others
- Home timeline shows recent posts from followed accounts, in reverse-chronological order
- Posts delivered to followers within ~5 seconds

**Nonfunctional**
- Sustain ~5,800 posts/s average, peak ~150,000 posts/s (events, breaking news)
- Serve home timeline reads with low latency (sub-second)
- Stay correct even with celebrity accounts (100M+ followers) and users following tens of thousands

### The two designs

```mermaid
flowchart LR
  subgraph Pull["Pull model (query at read)"]
    U1[Client polls every 5s] --> DB1[(posts JOIN follows<br/>ORDER BY time)]
  end
  subgraph Push["Push model (materialize on write)"]
    P[User posts] --> FanOut[Fan-out worker]
    FanOut --> T1[Timeline cache for follower 1]
    FanOut --> T2[Timeline cache for follower 2]
    FanOut --> Tn[... follower N]
    Reader[Client] --> T1
  end
```

### Back-of-envelope: pull vs push

In [ ]:
# Timeline fan-out estimation
# Assumptions from the chapter
POSTS_PER_DAY = 500_000_000
POSTS_PER_SECOND = POSTS_PER_DAY / 86_400  # ~5,787
CONCURRENT_USERS = 10_000_000               # users logged in / online
AVG_FOLLOWS = 200                           # people each user follows
AVG_FOLLOWERS = 200                         # mirror of the above (avg)
POLL_INTERVAL_S = 5                         # client polls every 5s

# --- Pull model: client re-runs the timeline query every 5s ---
# Each query must scan recent posts for the 200 accounts this user follows.
pull_queries_per_sec = CONCURRENT_USERS / POLL_INTERVAL_S  # 2M queries/s
pull_lookups_per_sec = pull_queries_per_sec * AVG_FOLLOWS   # 400M lookups/s

# --- Push model: on every post, write into each follower's timeline ---
push_writes_per_sec = POSTS_PER_SECOND * AVG_FOLLOWERS      # ~1.16M writes/s

print(f"Posts per second      : {POSTS_PER_SECOND:>15,.0f}")
print(f"Pull queries / sec    : {pull_queries_per_sec:>15,.0f}")
print(f"Pull LOOKUPS / sec    : {pull_lookups_per_sec:>15,.0f}  <-- dominated by joins")
print(f"Push WRITES / sec     : {push_writes_per_sec:>15,.0f}")
print(f"Savings ratio (pull/push): {pull_lookups_per_sec / push_writes_per_sec:.1f}x")

In [ ]:
# Where the push model breaks: celebrities
# A post by an account with N_followers triggers N_followers timeline writes.
# Justin Bieber / Obama scale:
CELEB_FOLLOWERS = 100_000_000
# How many avg-user posts would that one celebrity post equal, in fan-out work?
equivalent_posts = CELEB_FOLLOWERS / AVG_FOLLOWERS
print(f"One celebrity post = {equivalent_posts:,.0f} average-user posts of fan-out work.")

# Hybrid: store celebrity posts separately, merge at read time for their followers.
# If the top 0.1% of users are "celebs" (say > 1M followers), serve their posts via
# pull; serve everyone else's posts via push.
# This dramatically bounds the worst-case fan-out per write.

### Trade-offs

| Approach                  | Pros                                              | Cons                                                        | When to use                             |
|---------------------------|---------------------------------------------------|-------------------------------------------------------------|-----------------------------------------|
| **Pull** (query at read)  | Simple; no extra storage; fresh by construction   | Reads do huge joins; doesn't scale beyond a few M users     | Small scale, low read/write ratio       |
| **Push** (fan-out-on-write) | Reads cheap (served from cache); supports streaming | Writes do O(followers) work; celebrity posts are a hotspot | Typical social feed at scale            |
| **Hybrid**                | Bounds worst-case write fan-out; scales for celebs | More moving parts; read-time merge required                 | Skewed graphs with both heavy and light users |

> **See also**: `[[01-social-network-timeline-case-study]]`

## 2. Performance: Response Time, Percentiles, and Tail Latency

### Vocabulary (strict)

| Term              | Definition                                                                                      |
|-------------------|-------------------------------------------------------------------------------------------------|
| **Response time** | What the client sees — from send to fully-received response. Includes everything.               |
| **Service time**  | The portion the server is actively processing the request.                                      |
| **Queueing delay**| Time a request waits (CPU queue, outbound network buffer, etc.) before/after processing.        |
| **Network latency**| Time spent traveling on the wire.                                                              |
| **Latency (general)** | Catch-all for any time the request is *not* being actively processed (latent).             |

Queueing delays cause **head-of-line blocking**: a few slow requests hold up fast ones behind them. That's why response time must be measured client-side and reported as a distribution.

### The mean hides the tail — build the distribution yourself

In [ ]:
# Generate a realistic response-time sample using stdlib only.
# Log-normal: bulk of requests are fast, with a long right tail.
import random, math, statistics

random.seed(42)
N = 10_000
# mu/sigma chosen so that median ~ 200 ms and tail stretches into seconds
samples_ms = [math.exp(random.gauss(mu=math.log(200), sigma=0.8)) for _ in range(N)]

def percentile(sorted_xs, p):
    """p in [0, 100]. Linear interpolation between adjacent ranks."""
    if not sorted_xs:
        raise ValueError("empty")
    k = (len(sorted_xs) - 1) * (p / 100)
    lo, hi = int(math.floor(k)), int(math.ceil(k))
    if lo == hi:
        return sorted_xs[lo]
    return sorted_xs[lo] + (sorted_xs[hi] - sorted_xs[lo]) * (k - lo)

sorted_rt = sorted(samples_ms)
mean   = statistics.mean(samples_ms)
p50    = percentile(sorted_rt, 50)
p95    = percentile(sorted_rt, 95)
p99    = percentile(sorted_rt, 99)
p999   = percentile(sorted_rt, 99.9)
worst  = sorted_rt[-1]

print(f"N               = {N:,}")
print(f"mean (average)  = {mean:7.1f} ms")
print(f"p50  (median)   = {p50:7.1f} ms")
print(f"p95             = {p95:7.1f} ms")
print(f"p99             = {p99:7.1f} ms")
print(f"p999            = {p999:7.1f} ms")
print(f"worst           = {worst:7.1f} ms")
print()
print(f"Ratio p99 / p50 = {p99/p50:.1f}x   <- users in the tail feel a dramatically slower service")
print(f"Mean sits at percentile ~ {sum(1 for x in sorted_rt if x <= mean) / N * 100:.1f}%")

In [ ]:
# ASCII histogram so you can *see* the tail
buckets = [0] * 20
bucket_width_ms = 100  # each bucket = 100 ms
for x in samples_ms:
    idx = min(int(x // bucket_width_ms), len(buckets) - 1)
    buckets[idx] += 1

max_count = max(buckets)
print("Response-time histogram (each # ~= 1% of requests):")
for i, count in enumerate(buckets):
    lo = i * bucket_width_ms
    hi = lo + bucket_width_ms
    bar = "#" * (count * 50 // max_count)
    tag = f"{lo:4d}-{hi:4d} ms" if i < len(buckets) - 1 else f"{lo:4d}+    ms"
    print(f"  {tag} | {bar} {count}")

Notice the shape: **most requests are fast, but there's a long tail**. The mean is pulled right by the tail; it doesn't describe a "typical" user experience. The p99 and p999 describe the experience of your power users — often the ones with the most data and therefore the most slow-path work.

### Tail-latency amplification

Backend services are commonly called multiple times to serve one end-user request. Even if calls fan out in parallel, the end-user request waits for the **slowest** of them.

```mermaid
flowchart LR
  User -->|1 request| Gateway
  Gateway --> B1[Backend 1]
  Gateway --> B2[Backend 2]
  Gateway --> Bn[... Backend N]
  B1 --> Merge
  B2 --> Merge
  Bn --> Merge
  Merge --> User
```

In [ ]:
# If each backend call is slow with probability p_slow (i.e., exceeds the latency threshold),
# and they're independent, then the whole request is slow with probability:
#   P(slow user request) = 1 - (1 - p_slow) ** N

p_slow = 0.01          # each backend call exceeds threshold 1% of the time (p99 = threshold)
for N in (1, 5, 10, 50, 100, 500):
    p_user_slow = 1 - (1 - p_slow) ** N
    print(f"fan-out N={N:>3} backends  ->  P(end-user request is slow) = {p_user_slow:6.2%}")

**Read this carefully.** Each backend is "fast 99% of the time", but a 100-way fan-out means the user sees a slow request **63% of the time**. This is *why* Amazon monitors p999 for internal services: their slowest 0.1% sets the ceiling for everyone else.

> Corollary: **averaging percentiles across machines is wrong.** The right aggregation is to sum the histograms and recompute the percentile on the merged distribution.

> **See also**: `[[02-response-time-percentiles-and-tail-latency]]`

## 3. Metastable Failure and Overload Control

### The queueing knee

As throughput approaches capacity, response time grows **sharply** (not linearly). Past the knee, small additional load causes disproportionate latency spikes. If clients time out and retry, arriving load *increases*, driving the system deeper into overload. The system can remain stuck there even after the original trigger is gone. That is a **metastable failure**.

```mermaid
flowchart LR
  A[Normal load] --> B{Load spike or<br/>slow dependency}
  B --> C[Response times climb]
  C --> D[Clients time out]
  D --> E[Clients RETRY]
  E --> F[Effective load = original + retries]
  F --> C
  F --> G[System stays overloaded<br/>even after spike subsides]
  G -.reset / reboot.-> A
```

### Defenses (client-side and server-side)

| Where         | Technique            | What it does                                                           |
|---------------|----------------------|------------------------------------------------------------------------|
| Client        | **Exponential backoff + jitter** | Spread retries in time; don't synchronize retry storms.         |
| Client        | **Circuit breaker**  | Stop sending to an endpoint that has been failing; fail fast locally.  |
| Client        | **Token bucket** for retries | Cap total retry budget per unit time.                           |
| Server        | **Load shedding**    | Proactively reject requests when near capacity (fast 503 > slow 500).  |
| Server        | **Backpressure**     | Tell upstream to slow down (e.g., bounded queues, 429 + Retry-After).  |
| Both          | **Queue choice**     | Short bounded queues + fair scheduling prevent head-of-line blocking.  |

> **See also**: `[[03-metastable-failure-and-overload-control]]`

## 4. Reliability: Faults, Failures, and Fault Tolerance

- **Fault**: a single part stops working correctly (one disk dies, one VM reboots, one dependency times out).
- **Failure**: the system as a whole stops meeting its SLO.
- **Fault-tolerant** system: continues to provide service despite a bounded set of faults. The part whose fault becomes a failure is a **single point of failure (SPOF)**.

### Three families of faults

| Family       | Independence          | Examples                                                 | Typical mitigation                                |
|--------------|-----------------------|----------------------------------------------------------|---------------------------------------------------|
| **Hardware** | Mostly independent    | Disk 2–5%/yr, SSD 0.5–1%/yr, DRAM errors, CPU miscalcs   | Redundancy: RAID, dual PSUs, multiple AZs         |
| **Software** | **Highly correlated** | Leap-second kernel bug; firmware at 32,768 h; cascades   | Process isolation; crash-and-restart; chaos tests |
| **Human**    | Correlated by process | Config change rollouts; operator error                   | Canary/blue-green; rollback; blameless postmortems|

### MTBF / availability math

In [ ]:
# Fleet-level expected failures
DISKS = 10_000
ANNUAL_FAILURE_RATE = 0.02   # 2%/yr, within the chapter's 2-5% band for HDDs
annual_failures = DISKS * ANNUAL_FAILURE_RATE
daily_failures  = annual_failures / 365

print(f"Expected disk failures per year : {annual_failures:,.0f}")
print(f"Expected disk failures per day  : {daily_failures:.2f}")
print("  -> In a 10K-disk fleet you replace one nearly every day. Redundancy is not optional.\n")

In [ ]:
# Availability "nines" -> tolerable downtime per year
MIN_PER_YEAR = 365.25 * 24 * 60  # 525,960

def downtime(nines_pct):
    unavailable_fraction = 1 - nines_pct / 100
    total_min = unavailable_fraction * MIN_PER_YEAR
    if total_min >= 60:
        return f"{total_min/60:6.2f} hours/yr"
    return f"{total_min:6.2f} minutes/yr"

for n in [99, 99.9, 99.95, 99.99, 99.999]:
    print(f"  {n:>7.3f} %  ->  downtime = {downtime(n)}")

Three nines (99.9%) gives you ~8.76 hours/year of budgeted downtime. Four nines (99.99%) gives you ~52.6 minutes. Five nines gives you ~5.26 minutes — which is less than the time needed for one human to diagnose and fix something, so at five nines the recovery must be automated.

### Chaos engineering and blameless postmortems

```mermaid
flowchart LR
  Inject[Inject fault<br/>e.g. kill random node] --> Observe[Observe SLO impact]
  Observe --> Found{Did recovery<br/>work?}
  Found -- no --> Fix[Fix the gap]
  Fix --> Inject
  Found -- yes --> Confidence[Confidence in<br/>fault-tolerance machinery]
  Confidence --> Inject
```

Blameless postmortems ask "what about the sociotechnical system made this mistake possible?" — not "who should be punished?" Configuration changes by operators are the leading cause of outages; hardware plays a role in 10–25%.

> **See also**: `[[04-reliability-and-fault-tolerance]]`

## 5. Scalability: Shared-Memory, Shared-Disk, and Shared-Nothing

### The three architectures

| Axis                     | Shared-memory (vertical)         | Shared-disk                          | Shared-nothing (horizontal)           |
|--------------------------|----------------------------------|--------------------------------------|---------------------------------------|
| **Topology**             | One big box; threads share RAM   | Many CPUs over NAS/SAN               | Many nodes, each with own CPU/RAM/disk|
| **Fault tolerance**      | Whole box is a SPOF              | Disk array is shared; locking limits | Nodes fail independently              |
| **Cost curve**           | Super-linear (big boxes expensive) | Moderate, but locking overhead      | Closest to linear in the cloud        |
| **Elasticity**           | Reboot to resize                 | Limited                              | Add/remove nodes online               |
| **Coordination**         | In-process                       | Lock manager over network            | Explicit sharding + network protocols |
| **Fits which workloads** | Small, latency-critical          | Legacy DW, moderate OLTP             | Internet-scale; cloud-native          |

A cloud-native variant **separates storage and compute**: multiple stateless compute nodes share a purpose-built storage service (not a generic filesystem or block device). This hybrid inherits shared-disk's operational simplicity while avoiding its locking bottlenecks.

```mermaid
flowchart TB
  subgraph SM["Shared-memory"]
    CPU1 --- RAM[(RAM)]
    CPU2 --- RAM
    CPUn --- RAM
    RAM --- Disks1[(Disks)]
  end
  subgraph SD["Shared-disk"]
    N1[Node] --- Net1[Fast net]
    N2[Node] --- Net1
    N3[Node] --- Net1
    Net1 --- SAN[(Shared SAN / NAS)]
  end
  subgraph SN["Shared-nothing"]
    M1[Node + local disk] --- Net2[Network]
    M2[Node + local disk] --- Net2
    M3[Node + local disk] --- Net2
  end
```

### Principles for scalability

- **No "magic scaling sauce"** — the right architecture depends on the workload. A system doing 100k req/s at 1kB each looks nothing like a system doing 3 req/min at 2 GB each, even though throughput is the same.
- **Every order of magnitude in load forces a rethink.** Don't plan more than ~10x ahead.
- **Split into components that can operate independently.** This is the insight behind microservices, sharding, stream processing, and shared-nothing itself.
- **Don't make it more complicated than it needs to be.** A single-box database often beats a shaky distributed one. Manually-scaled predictable systems often beat autoscalers you don't understand.

> **See also**: `[[05-scaling-architectures-shared-nothing]]`

## 6. Maintainability: Operability, Simplicity, Evolvability

The majority of software cost is **maintenance**, not initial development. Design the system so that *future you* (and the engineers who've never met you) can keep it running, understand it, and change it.

| Pillar            | What it means                                                              | Failure mode                                   | Practices                                                        |
|-------------------|----------------------------------------------------------------------------|------------------------------------------------|-------------------------------------------------------------------|
| **Operability**   | Make routine operations easy; surface the system's health                  | Every incident requires heroics from on-call   | Good monitoring/observability; rolling upgrades; predictable defaults; self-healing with manual override |
| **Simplicity**    | Manage complexity with good abstractions                                   | "Big ball of mud"; hidden assumptions; bugs on change | Prefer well-understood patterns; keep essential complexity, delete accidental complexity; design patterns, DDD |
| **Evolvability**  | Make change easy; minimize irreversibility                                 | One-way doors block future pivots              | Loose coupling; TDD + refactoring; feature flags; reversible migrations; Agile at the system level |

- Essential vs. accidental complexity is a useful lens, but the boundary shifts as tooling improves — what was accidental in 2010 may be essential in 2025 (and vice versa).
- Abstractions (SQL, high-level languages, event logs, transactions) are the main tool for managing complexity: reuse gets both efficiency and quality.
- Irreversibility compounds: a migration you can't undo carries much higher stakes than one you can. Keep the back door open.

> **See also**: `[[06-maintainability-operability-simplicity-evolvability]]`

## Self-Check

Work through these before moving on.

1. Why do we use **p99** or **p999** for SLOs rather than the **mean** response time? Give one concrete reason rooted in the tail.
2. Your service has p99 = 200 ms per backend call. A user request fans out to 20 backends. What fraction of user requests see a slow (>200 ms) call? (Hint: `1 - 0.99**20`.)
3. In the social-network case study, why is the **push** model preferred at scale despite doing more work on writes? Where does it break down, and what's the hybrid fix?
4. Distinguish **fault** from **failure**. Why is an SPOF the one place where the distinction collapses?
5. A cluster of 5,000 SSDs has an annual failure rate of 1%. How many expected failures per week? What does that imply for automation vs. manual replacement?
6. A vendor promises "four nines." How much downtime per year does that allow? Is human-in-the-loop recovery compatible with five nines?
7. Name three defenses against **metastable failure**, one on the client and two on the server.
8. Why does **shared-disk** not scale as cleanly as **shared-nothing** in practice? What modern variant combines the ops simplicity of the former with the scalability of the latter?
9. Under the **essential vs. accidental** framing, give one example of complexity in your own codebase that is arguably accidental today. What would make it essential? What would eliminate it?
10. Why are **blameless** postmortems more effective than "human error" root-causes, according to the chapter?

## See Also

- `[[01-social-network-timeline-case-study]]` — pull vs push vs hybrid fan-out
- `[[02-response-time-percentiles-and-tail-latency]]` — percentiles, tail amplification, SLOs
- `[[03-metastable-failure-and-overload-control]]` — retry storms, backpressure, circuit breakers
- `[[04-reliability-and-fault-tolerance]]` — faults, failures, chaos engineering, blameless postmortems
- `[[05-scaling-architectures-shared-nothing]]` — shared-memory, shared-disk, shared-nothing, storage/compute separation
- `[[06-maintainability-operability-simplicity-evolvability]]` — the three maintainability pillars